# 🤖 AI Functions Showcase - The Art of the Possible

**COMPREHENSIVE AI DEMONSTRATION** - Run this after notebook 01.

## What this notebook demonstrates:
- ✅ **ai_classify** - Intelligent priority and category classification
- ✅ **ai_extract** - Structured data extraction using entity labels
- ✅ **ai_gen** - Complex analysis, summaries, and creative content generation
- ✅ **Final Summary Table** - Complete AI-powered ticket analysis

**Prerequisites:** Run `01_sample_data_generation.ipynb` first

## AI Functions Showcase:
- `ai_classify` - For priority and category classification
- `ai_extract` - For structured data extraction using entity labels
- `ai_gen` - For complex analysis and content generation

---

## 🎯 Goal: Demonstrate the full power of Databricks AI Functions


In [1]:
# Import required libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json
from datetime import datetime

# Import configuration
%run ./config

print("✅ Libraries imported and configuration loaded")
print(f"🎯 Using Unity Catalog: {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}")
print("🚀 Ready to showcase Databricks AI Functions!")


✅ Libraries imported and configuration loaded
🎯 Using Unity Catalog: quickstart_catalog_vkm_external.classify_tickets
🚀 Ready to showcase Databricks AI Functions!


In [2]:
# Load Sample Data
print("📊 Loading sample ticket data from Unity Catalog...")

# Load the sample ticket data from Unity Catalog
df_tickets = spark.table(TABLES["raw_tickets"])
print(f"✅ Loaded {df_tickets.count()} tickets from: {TABLES['raw_tickets']}")

# Display sample data
print("\n📋 Sample ticket data:")
display(df_tickets.select("ticket_id", "short_description", "description").limit(3))


📊 Loading sample ticket data from Unity Catalog...


✅ Loaded 64 tickets from: quickstart_catalog_vkm_external.classify_tickets.raw_tickets

📋 Sample ticket data:


,ticket_id,short_description,description
0,TICKET_01_001,Issue #1-1 - Important,customers are reporting that their data is missing after the last update. this is a big problem and we need to investigate immediately. could be a data migration issue.
1,TICKET_01_002,Issue #1-2 - Urgent,the chat support widget disappeared from our website. customers can't reach support and they're complaining on social media.
2,TICKET_01_003,Issue #1-3 - Problem,the deployment failed again and now production is down. we rolled back but need to figure out what went wrong. this is the third time this month. we need better testing.


# 🎯 AI Function #1: ai_classify

**Purpose:** Intelligent classification of text into predefined categories

**Use Cases:** Priority classification, category assignment, sentiment analysis, risk assessment

**Example:** Classify ticket priorities based on description content


In [3]:
# AI Function #1: ai_classify - Priority Classification
print("🎯 Demonstrating ai_classify for priority classification...")

# Register DataFrame as temporary view for SQL access
df_tickets.createOrReplaceTempView("tickets")

# Use ai_classify to classify ticket priorities
df_with_priority = spark.sql("""
    SELECT
        *,
        ai_classify(
            description, 
            ARRAY('Low Priority', 'Medium Priority', 'High Priority', 'Urgent Priority')
        ) as ai_priority_classification
    FROM tickets
""")

print("✅ ai_classify completed - Priority classification done!")
print("\n📊 Priority Classification Results:")
display(df_with_priority.select("ticket_id", "short_description", "ai_priority_classification").limit(5))

# Show distribution of AI classifications
print("\n📈 Priority Distribution:")
df_with_priority.groupBy("ai_priority_classification").count().orderBy(desc("count")).show()


🎯 Demonstrating ai_classify for priority classification...
✅ ai_classify completed - Priority classification done!

📊 Priority Classification Results:


,ticket_id,short_description,ai_priority_classification
0,TICKET_01_001,Issue #1-1 - Important,Urgent Priority
1,TICKET_01_002,Issue #1-2 - Urgent,Urgent Priority
2,TICKET_01_003,Issue #1-3 - Problem,High Priority
3,TICKET_01_004,Issue #1-4 - Important,High Priority
4,TICKET_01_005,Issue #1-5 - Help needed,Urgent Priority



📈 Priority Distribution:


+--------------------------+-----+
|ai_priority_classification|count|
+--------------------------+-----+
|           Urgent Priority|   38|
|             High Priority|   22|
|           Medium Priority|    4|
+--------------------------+-----+



# 🎯 AI Function #2: ai_extract

**Purpose:** Extract structured data from unstructured text using entity labels

**Use Cases:** Action items extraction, requirement identification, urgency assessment, system identification

**Example:** Extract specific entities (action_items, main_requirement, urgency_level, affected_systems) from ticket descriptions


In [4]:
# AI Function #2: ai_extract - Structured Data Extraction
print("🎯 Demonstrating ai_extract for structured data extraction...")

# Register DataFrame as temporary view for SQL access
df_with_priority.createOrReplaceTempView("tickets_with_priority")

# Use ai_extract with correct ARRAY<STRING> syntax for labels
df_with_extraction = spark.sql("""
    SELECT
        *,
        ai_extract(
            description,
            ARRAY('action_items', 'main_requirement', 'urgency_level', 'affected_systems')
        ) as ai_extracted_data
    FROM tickets_with_priority
""")

print("✅ ai_extract completed - Structured data extraction done!")
print("\n📊 Extraction Results:")
display(df_with_extraction.select("ticket_id", "short_description", "ai_extracted_data").limit(3))

# Parse the extracted data for better display
df_parsed = df_with_extraction.withColumn(
    "action_items", 
    col("ai_extracted_data.action_items")
).withColumn(
    "main_requirement", 
    col("ai_extracted_data.main_requirement")
).withColumn(
    "urgency_level", 
    col("ai_extracted_data.urgency_level")
).withColumn(
    "affected_systems", 
    col("ai_extracted_data.affected_systems")
)

print("\n📋 Parsed Extraction Results:")
display(df_parsed.select("ticket_id", "action_items", "main_requirement", "urgency_level").limit(3))


🎯 Demonstrating ai_extract for structured data extraction...
✅ ai_extract completed - Structured data extraction done!

📊 Extraction Results:


,ticket_id,short_description,ai_extracted_data
0,TICKET_01_001,Issue #1-1 - Important,"{'action_items': 'investigate', 'main_requirement': 'investigate immediately', 'urgency_level': 'immediately', 'affected_systems': 'data migration'}"
1,TICKET_01_002,Issue #1-2 - Urgent,"{'action_items': 'restore the chat support widget', 'main_requirement': 'restore the functionality of the chat support widget', 'urgency_level': 'high', 'affected_systems': 'chat support widget'}"
2,TICKET_01_003,Issue #1-3 - Problem,"{'action_items': 'figure out what went wrong and need better testing', 'main_requirement': 'need better testing', 'urgency_level': None, 'affected_systems': 'production'}"



📋 Parsed Extraction Results:


,ticket_id,action_items,main_requirement,urgency_level
0,TICKET_01_001,investigate,investigate immediately,immediately
1,TICKET_01_002,restore the chat support widget,restore the functionality of the chat support widget,high
2,TICKET_01_003,figure out what went wrong and need better testing,need better testing,None


# 🎯 AI Function #3: ai_gen

**Purpose:** Generate creative content, summaries, and complex analysis

**Use Cases:** Executive summaries, detailed analysis, creative content, recommendations

**Example:** Generate comprehensive ticket analysis and recommendations


In [5]:
# AI Function #3: ai_gen - Comprehensive Analysis
print("🎯 Demonstrating ai_gen for comprehensive analysis...")

# Register DataFrame as temporary view for SQL access
df_parsed.createOrReplaceTempView("tickets_with_extraction")

# Use ai_gen to create comprehensive analysis
df_with_analysis = spark.sql("""
    SELECT
        *,
        ai_gen(
            CONCAT(
                'Analyze this IT ticket and provide: ',
                '1. Executive summary (2-3 sentences), ',
                '2. Technical complexity (Low/Medium/High), ',
                '3. Estimated effort (hours), ',
                '4. Risk assessment (Low/Medium/High), ',
                '5. Recommended next steps. ',
                'Ticket: ', short_description, ' - ', description
            )
        ) as ai_comprehensive_analysis
    FROM tickets_with_extraction
""")

print("✅ ai_gen completed - Comprehensive analysis done!")
print("\n📊 Analysis Results:")
display(df_with_analysis.select("ticket_id", "short_description", "ai_comprehensive_analysis").limit(3))


🎯 Demonstrating ai_gen for comprehensive analysis...
✅ ai_gen completed - Comprehensive analysis done!

📊 Analysis Results:


,ticket_id,short_description,ai_comprehensive_analysis
0,TICKET_01_001,Issue #1-1 - Important,"Here is the analysis of the IT ticket:\n\n1. **Executive Summary**: Customers are reporting missing data after the last update, indicating a potential data migration issue that requires immediate investigation. This issue has significant impact on customer experience and data integrity. Prompt resolution is necessary to mitigate potential losses and reputational damage.\n2. **Technical Complexity**: High - The issue involves data migration, which can be a complex process, and the fact that customers are reporting missing data suggests that the problem may be widespread and require a thorough investigation to identify the root cause.\n3. **Estimated Effort**: 8-12 hours - Depending on the scope of the issue and the complexity of the data migration process, it may take several hours to investigate, identify the root cause, and develop a plan to restore the missing data.\n4. **Risk Assessment**: High - The loss of customer data can have significant consequences, including reputational damage, financial losses, and potential regulatory issues. The longer it takes to resolve the issue, the higher the risk of these consequences.\n5. **Recommended Next Steps**: \n* Immediately assemble a team to investigate the issue, including representatives from development, QA, and data management.\n* Gather more information about the missing data, including the scope of the issue and the affected customers.\n* Review the data migration process and logs to identify potential errors or issues.\n* Develop a plan to restore the missing data and prevent similar issues in the future.\n* Provide regular updates to stakeholders and customers on the progress of the investigation and resolution."
1,TICKET_01_002,Issue #1-2 - Urgent,"Here's the analysis of the IT ticket:\n\n**1. Executive Summary**: The chat support widget has disappeared from the company's website, preventing customers from reaching support and resulting in complaints on social media. This issue is urgent and requires immediate attention to minimize the impact on customer experience and reputation. The goal is to restore the chat support widget as soon as possible.\n\n**2. Technical Complexity**: Medium - The issue is likely related to a configuration or integration problem with the chat support software, which may require some technical investigation to resolve.\n\n**3. Estimated Effort**: 2-4 hours - Depending on the complexity of the issue, it may take a few hours to investigate, identify, and fix the problem. This estimate assumes that the issue is related to a configuration or integration problem and not a more complex technical issue.\n\n**4. Risk Assessment**: High - The disappearance of the chat support widget is having a direct impact on customer experience and reputation, with customers already complaining on social media. If not resolved quickly, this issue could lead to a loss of customer trust and potential revenue.\n\n**5. Recommended Next Steps**: \n* Immediately investigate the issue to determine the cause of the problem.\n* Check the chat support software configuration, integration, and any recent changes that may have caused the issue.\n* If necessary, engage with the chat support software vendor or a technical expert to assist with the investigation and resolution.\n* Once the issue is resolved, verify that the chat support widget is functioning correctly and monitor customer feedback to ensure that the issue is fully resolved."
2,TICKET_01_003,Issue #1-3 - Problem,"Here is the analysis of the IT ticket:\n\n1. **Executive Summary**: The recent deployment failed, causing production to go down, and a rollback was necessary. This is the third deployment failure this month, indicating a recurring issue that needs to be addressed. The team requires a thorough investigation and improved testing to prevent future failures.\n2. **Technical Complexity**: Medium - The issue involves dep

# 🎉 Final Summary Table - The Art of the Possible

**Complete AI-powered ticket analysis showcasing all three AI functions**


In [6]:
# Create Final Summary Table
print("🎉 Creating comprehensive summary table...")

# Create a clean summary table with all AI results
df_final_summary = df_with_analysis.select(
    "ticket_id",
    "short_description",
    "ai_priority_classification",
    "action_items",
    "main_requirement", 
    "urgency_level",
    "affected_systems",
    "ai_comprehensive_analysis"
).withColumn(
    "ai_showcase_timestamp", 
    current_timestamp()
)

print("✅ Final summary table created!")
print(f"📊 Total tickets analyzed: {df_final_summary.count()}")

# Display the comprehensive results
print("\n🎯 COMPLETE AI SHOWCASE RESULTS:")
print("="*80)
display(df_final_summary.limit(5))

# Save to Unity Catalog
print("\n💾 Saving results to Unity Catalog...")
df_final_summary.write.format("delta").mode("overwrite").saveAsTable(TABLES["ai_showcase_results"])
print(f"✅ Results saved to: {TABLES['ai_showcase_results']}")

# Show summary statistics
print("\n📈 AI Showcase Summary Statistics:")
print(f"🎯 Tickets processed: {df_final_summary.count()}")
print(f"🤖 AI functions demonstrated: 3 (ai_classify, ai_extract, ai_gen)")
print(f"📊 Data saved to Unity Catalog: {TABLES['ai_showcase_results']}")

print("\n" + "="*80)
print("🎉 AI SHOWCASE COMPLETED SUCCESSFULLY!")
print("="*80)
print("✅ ai_classify: Priority classification")
print("✅ ai_extract: Structured data extraction using entity labels")
print("✅ ai_gen: Comprehensive analysis and content generation")
print("✅ Final summary table created and saved")
print("="*80)


🎉 Creating comprehensive summary table...
✅ Final summary table created!


📊 Total tickets analyzed: 64

🎯 COMPLETE AI SHOWCASE RESULTS:


,ticket_id,short_description,ai_priority_classification,action_items,main_requirement,urgency_level,affected_systems,ai_comprehensive_analysis,ai_showcase_timestamp
0,TICKET_01_001,Issue #1-1 - Important,Urgent Priority,investigate,investigate immediately,big problem,data migration issue,"Here is the analysis of the IT ticket:\n\n1. **Executive Summary**: Customers are reporting missing data after the last update, indicating a potential data migration issue that requires immediate investigation. This issue has significant impact on customer experience and data integrity. Prompt resolution is necessary to mitigate potential losses and reputational damage.\n2. **Technical Complexity**: High - The issue involves data migration, which can be a complex process, and the fact that customers are reporting missing data suggests that the problem may be widespread and require a thorough investigation to identify the root cause.\n3. **Estimated Effort**: 8-12 hours - Depending on the scope of the issue and the complexity of the data migration process, it may take several hours to investigate, identify the root cause, and develop a plan to restore the missing data.\n4. **Risk Assessment**: High - The loss of customer data can have significant consequences, including reputational damage, financial losses, and potential regulatory issues. The longer it takes to resolve the issue, the higher the risk of these consequences.\n5. **Recommended Next Steps**: \n* Immediately assemble a team to investigate the issue, including representatives from development, QA, and data management.\n* Gather more information about the missing data, including the scope of the issue and the affected customers.\n* Review the data migration process and logs to identify potential errors or issues.\n* Develop a plan to restore the missing data and prevent similar issues in the future.\n* Provide regular updates to stakeholders and customers on the progress of the investigation and resolution.",2025-09-21 01:57:06.077108
1,TICKET_01_002,Issue #1-2 - Urgent,Urgent Priority,restore the chat support widget,restore the functionality of the chat support widget,high,chat support widget,"Here's the analysis of the IT ticket:\n\n**1. Executive Summary**: The chat support widget has disappeared from the company's website, preventing customers from reaching support and resulting in complaints on social media. This issue is urgent and requires immediate attention to minimize the impact on customer experience and reputation. The goal is to restore the chat support widget as soon as possible.\n\n**2. Technical Complexity**: Medium - The issue is likely related to a configuration or integration problem with the chat support software, which may require some technical investigation to resolve.\n\n**3. Estimated Effort**: 2-4 hours - Depending on the complexity of the issue, it may take a few hours to investigate, identify, and fix the problem. This estimate assumes that the issue is related to a configuration or integration problem and not a more complex technical issue.\n\n**4. Risk Assessment**: High - The disappearance of the chat support widget is having a direct impact on customer experience and reputation, with customers already complaining on social media. If not resolved quickly, this issue could lead to a loss of customer trust and potential revenue.\n\n**5. Recommended Next Steps**: \n* Immediately investigate the issue to determine the cause of the problem.\n* Check the chat support software configuration, integration, and any recent changes that may have caused the issue.\n* If necessary, engage with the chat support software vendor or a technical expert to assist with the investigation and resolution.\n* Once the issue is resolved, verify that the chat support widget is functioning correctly and monitor customer feedback to ensure that the issue is fully resolved.",2025-09-21 01:57:06.077108
2,TICKET_01_003,Issue #1-3 - Problem,High Priority,figure out what went wrong and


💾 Saving results to Unity Catalog...


✅ Results saved to: quickstart_catalog_vkm_external.classify_tickets.ai_showcase_results

📈 AI Showcase Summary Statistics:


🎯 Tickets processed: 64
🤖 AI functions demonstrated: 3 (ai_classify, ai_extract, ai_gen)
📊 Data saved to Unity Catalog: quickstart_catalog_vkm_external.classify_tickets.ai_showcase_results

🎉 AI SHOWCASE COMPLETED SUCCESSFULLY!
✅ ai_classify: Priority classification
✅ ai_extract: Structured data extraction using entity labels
✅ ai_gen: Comprehensive analysis and content generation
✅ Final summary table created and saved
